# On-Chain BTC Direction Signals

Predict **BTC price-direction regimes** from *on-chain* fundamentals: network
security, miner economics, holder profitability, and whale/transfer flow.

**Thesis.** On-chain data is the ground truth of Bitcoin's supply/demand
structure. Miners must sell to cover fiat costs (revenue/hashrate = *hashprice*
dictates their profitability and capitulation risk); the **Puell Multiple**
(miner revenue vs. its 1-year average) times miner stress. Holder profitability
metrics (**MVRV**, **SOPR**, **NUPL**) mark cycle tops/bottoms: when everyone is
in deep profit, supply overhang builds; when coins move at a loss, capitulation
is near. Network activity (active addresses, tx count, transfer volume) leads
adoption cycles. If we (a) measure these structural states, (b) detect which
*lead* price, and (c) composite them into a cycle-valuation regime, we get a
long-horizon directional edge that is orthogonal to the macro-liquidity view in
`Macro.ipynb`.

**What this notebook does**
1. Set up a **local on-chain DuckDB store** (`data/onchain.duckdb`) with an
   **incremental, rate-limited sync workflow** — re-fetch only when stale, sleep
   between requests, idempotent upserts. No IP bans, no terms violations.
2. Pull **10 keyless historical series** from `blockchain.info` (hashrate,
   difficulty, miner revenue, fees, active addresses, tx count, transfer volume,
   market cap, supply, cost-per-tx-%) — no key required.
3. Pull **premium valuation indicators** from **Glassnode** (MVRV, SOPR, NUPL,
   realized price, RHODL, exchange balance) — optional `GLASSNODE_API_KEY`.
   Without it the notebook degrades to *plausible synthetic* paths so the full
   pipeline still runs end-to-end. **Set the key for research-grade signals.**
4. Engineer on-chain features: **hashprice** (miner profitability), **Puell
   Multiple**, fee-revenue share, YoY activity growth, transfer velocity, plus
   the Glassnode valuation ratios.
5. **Lead/lag analysis** — cross-correlation + Granger vs. forward 13-week BTC
   returns to find which on-chain signals lead price.
6. Build an **on-chain cycle-valuation composite** and a bullish/bearish regime.
7. **Backtest** forward BTC returns (1m / 3m / 6m / 12m) conditional on regime
   and on early-warn bullish triggers.
8. Fit a **Probit** model for *P(bullish regime in ~90 days)* scored
   walk-forward (Brier score).
9. Dashboard + snapshot of the current on-chain state.

> **Run top-to-bottom.** Keys live in `.env` (see `config/example.yaml` for the
> shape). Without keys the notebook degrades gracefully to synthetic series —
> **set `GLASSNODE_API_KEY` for research-grade valuation signals.** The
> blockchain.info series are keyless and always live.


In [1]:
from __future__ import annotations

import os
import time
import warnings
from datetime import UTC, date, datetime
from pathlib import Path
from typing import Any

import duckdb
import httpx
import numpy as np
import plotly.graph_objects as go
import polars as pl
import statsmodels.api as sm
from dotenv import load_dotenv
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import grangercausalitytests

from ccquant.forecasting import load_daily_panel

warnings.filterwarnings("ignore", category=FutureWarning)

# --- Config ----------------------------------------------------------------
_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root.parent != _root:
    _root = _root.parent

load_dotenv(_root / ".env")

# Price/OHLCV store (from ccquant sync)
DB_PATH = os.environ.get("CCQUANT_DB", str(_root / "data" / "ccquant.duckdb"))
# Dedicated on-chain store — keeps on-chain series out of the OHLCV schema
ONCHAIN_DB = os.environ.get("CCQUANT_ONCHAIN_DB", str(_root / "data" / "onchain.duckdb"))

GLASSNODE_API_KEY = os.environ.get("GLASSNODE_API_KEY", "")

WEEKLY_FREQ = "1w"
LOOKBACK_YOY = 52            # weeks — YoY-style on-chain growth
MOM_LOOKBACK = 12            # weeks — momentum / early-signal window
LEAD_MIN, LEAD_MAX = -26, 26     # cross-correlation lag range (weeks)
FWD_HORIZONS_WK = [4, 13, 26, 52]  # 1m / 3m / 6m / 12m forward returns
PROBIT_HORIZON_WK = 13           # predict bullish regime within ~90d

# --- Rate-limit / good-citizen settings ------------------------------------
# blockchain.info is keyless with no published hard limit; stay gentle.
BC_INFO_REQUEST_DELAY = 1.0      # seconds between blockchain.info requests
# Glassnode Light API (free/Advanced tier): ~50 calls/day, 14d history.
GLASSNODE_REQUEST_DELAY = 6.0    # seconds between Glassnode requests
REFRESH_STALE_HOURS = 12         # daily series: only refetch if older than this
RNG_SEED = 11

print(f"OHLCV DB:        {DB_PATH}")
print(f"On-chain DB:     {ONCHAIN_DB}")
print(f"GLASSNODE key:   {'set' if GLASSNODE_API_KEY else 'NOT set (synthetic fallback)'}")
print(f"Lead/lag range:  {LEAD_MIN}..{LEAD_MAX} weeks")


OHLCV DB:        /home/ricka/Git/GitHub/ccquant/data/ccquant.duckdb
On-chain DB:     /home/ricka/Git/GitHub/ccquant/data/onchain.duckdb
GLASSNODE key:   NOT set (synthetic fallback)
Lead/lag range:  -26..26 weeks


## 1. Local on-chain store + rate-limited incremental sync

A dedicated DuckDB database (`data/onchain.duckdb`, override via
`CCQUANT_ONCHAIN_DB`) holds on-chain series in a tidy **long format**
(`metric`, `date`, `value`, `source`) plus a per-metric `onchain_sync_state`
table for incremental refresh.

**The sync workflow is designed to avoid IP bans / rate limits:**
- **Incremental:** each metric records `last_refresh_at`; a metric is only
  re-fetched when its data is older than `REFRESH_STALE_HOURS` (12h for daily
  series) — so re-running the notebook hits the network only for stale metrics.
- **Throttled:** a configurable `time.sleep` runs *between every* request
  (`BC_INFO_REQUEST_DELAY`, `GLASSNODE_REQUEST_DELAY`).
- **Idempotent:** upserts on `(metric, date, source)` make re-runs safe.
- **Caching:** once in DuckDB, all downstream analysis reads locally — zero
  repeated network calls. Pass `force=True` to ignore staleness and refetch all.
- **Respectful:** one request at a time, no concurrency, sensible timeouts. This
  stays well within blockchain.info's informal tolerances and the Glassnode
  Light API's ~50 calls/day cap.


In [2]:
class OnChainStore:
    """Tiny DuckDB wrapper for on-chain time series in long (tidy) format."""

    def __init__(self, path: str | Path) -> None:
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.conn = duckdb.connect(str(self.path))
        self.init_schema()

    def init_schema(self) -> None:
        self.conn.execute(
            """
            create table if not exists onchain_series (
              metric varchar not null,
              date   date    not null,
              value  double  not null,
              source varchar not null,
              primary key (metric, date, source)
            )
            """
        )
        self.conn.execute(
            """
            create table if not exists onchain_sync_state (
              metric           varchar not null,
              source           varchar not null,
              latest_at        varchar,
              last_refresh_at  timestamp,
              primary key (metric, source)
            )
            """
        )

    def upsert_series(self, metric: str, rows: list[tuple[date, float]], source: str) -> int:
        for d, v in rows:
            self.conn.execute(
                """insert into onchain_series (metric, date, value, source)
                   values (?, ?, ?, ?)
                   on conflict (metric, date, source) do update set value = excluded.value""",
                [metric, d, float(v), source],
            )
        return len(rows)

    def get_state(self, metric: str, source: str) -> tuple[str | None, datetime | None]:
        row = self.conn.execute(
            "select latest_at, last_refresh_at from onchain_sync_state where metric=? and source=?",
            [metric, source],
        ).fetchone()
        if row is None:
            return None, None
        lr = row[1]
        if lr is not None:
            if not isinstance(lr, datetime):
                lr = datetime.fromisoformat(str(lr))
            if lr.tzinfo is None:
                lr = lr.replace(tzinfo=UTC)
        return (str(row[0]) if row[0] is not None else None), lr

    def upsert_state(self, metric: str, source: str, latest_at: date | None) -> None:
        self.conn.execute(
            """insert into onchain_sync_state (metric, source, latest_at, last_refresh_at)
               values (?, ?, ?, ?)
               on conflict (metric, source) do update set
                 latest_at = excluded.latest_at,
                 last_refresh_at = excluded.last_refresh_at""",
            [metric, source, latest_at.isoformat() if latest_at else None, datetime.now(UTC)],
        )

    def load_series(self, metric: str) -> pl.DataFrame:
        df = pl.from_arrow(
            self.conn.execute(
                "select date, value from onchain_series where metric=? order by date", [metric]
            ).to_arrow_table()
        )
        return df if isinstance(df, pl.DataFrame) else df.to_frame()

    def metrics_present(self) -> list[str]:
        rows = self.conn.execute("select distinct metric from onchain_series").fetchall()
        return [str(r[0]) for r in rows]

    def close(self) -> None:
        self.conn.close()


store = OnChainStore(ONCHAIN_DB)
print(f"On-chain store: {ONCHAIN_DB}")
print(f"Existing metrics: {store.metrics_present()}")


On-chain store: /home/ricka/Git/GitHub/ccquant/data/onchain.duckdb
Existing metrics: []


### 1a. blockchain.info keyless historical series (no key, full history)

All series use the same charts API: `api.blockchain.info/charts/<chart>?timespan=all&format=json`
returning `{"values": [{"x": unix_ts, "y": value}, ...]}`. Keyless, generous
history, daily resolution. We fetch the full history once per metric and let the
sync state keep subsequent runs cheap.


In [3]:
# chart-id -> (our metric name, human label)
BLOCKCHAIN_METRICS: dict[str, dict[str, str]] = {
    "hash-rate":                       {"metric": "hashrate",            "label": "Network hashrate (TH/s)"},
    "difficulty":                      {"metric": "difficulty",          "label": "Mining difficulty"},
    "miners-revenue":                  {"metric": "miner_revenue_usd",   "label": "Miner revenue (USD/day)"},
    "transaction-fees-usd":            {"metric": "fees_usd",            "label": "Tx fees (USD/day)"},
    "n-unique-addresses":              {"metric": "active_addresses",    "label": "Active addresses"},
    "n-transactions":                  {"metric": "tx_count",            "label": "Transaction count"},
    "estimated-transaction-volume-usd":{"metric": "transfer_volume_usd", "label": "Est. transfer volume (USD)"},
    "market-cap":                      {"metric": "market_cap",          "label": "Market cap (USD)"},
    "total-bitcoins":                  {"metric": "supply",              "label": "Circulating supply (BTC)"},
    "cost-per-transaction-percent":    {"metric": "cost_per_tx_pct",     "label": "Miner cost / tx (%)"},
}

BC_API = "https://api.blockchain.info/charts"


def fetch_blockchain_chart(chart: str, timespan: str = "all") -> list[tuple[date, float]]:
    """Fetch a blockchain.info chart as [(date, value), ...]. Retries once on 429."""
    url = f"{BC_API}/{chart}"
    for _attempt in range(2):
        with httpx.Client(timeout=30.0) as client:
            resp = client.get(url, params={"timespan": timespan, "format": "json"})
        if resp.status_code == 429:
            time.sleep(60)
            continue
        resp.raise_for_status()
        vals = resp.json()["values"]
        return [
            (datetime.fromtimestamp(int(v["x"]), tz=UTC).date(), float(v["y"]))
            for v in vals if v.get("y") is not None
        ]
    return []


def sync_blockchain(force: bool = False) -> None:
    """Incremental, rate-limited sync of all blockchain.info metrics."""
    now = datetime.now(UTC)
    print(f"Syncing blockchain.info ({len(BLOCKCHAIN_METRICS)} metrics) "
          f"{'[FORCE]' if force else '[stale-only]'} ...")
    for chart, meta in BLOCKCHAIN_METRICS.items():
        metric = meta["metric"]
        _, last_refresh = store.get_state(metric, "blockchain.info")
        if not force and last_refresh is not None:
            age_h = (now - last_refresh).total_seconds() / 3600
            if age_h < REFRESH_STALE_HOURS:
                print(f"  {metric:>22}: fresh ({age_h:.1f}h ago) — skip")
                continue
        try:
            rows = fetch_blockchain_chart(chart, timespan="all")
        except Exception as exc:
            print(f"  {metric:>22}: FAILED ({exc})")
            continue
        n = store.upsert_series(metric, rows, "blockchain.info")
        latest = max(d for d, _ in rows) if rows else None
        store.upsert_state(metric, "blockchain.info", latest)
        print(f"  {metric:>22}: {n:>5} rows  up to {latest}")
        time.sleep(BC_INFO_REQUEST_DELAY)
    print("blockchain.info sync done.")


sync_blockchain()


Syncing blockchain.info (10 metrics) [stale-only] ...
                hashrate:  1599 rows  up to 2026-07-08
              difficulty:  1599 rows  up to 2026-07-08
       miner_revenue_usd:  1596 rows  up to 2026-07-07
                fees_usd:  1596 rows  up to 2026-07-07
        active_addresses:  1592 rows  up to 2026-07-06
                tx_count:  1595 rows  up to 2026-07-06
     transfer_volume_usd:  1444 rows  up to 2026-07-09
              market_cap:  1501 rows  up to 2026-07-03
                  supply:  1501 rows  up to 2026-07-03
         cost_per_tx_pct:  1444 rows  up to 2026-07-09
blockchain.info sync done.


### 1b. Glassnode premium valuation indicators (optional `GLASSNODE_API_KEY`)

These are the *cycle-valuation* metrics that blockchain.info does not provide:
- **MVRV** — market-value to realized-value (cheap/expensive vs. cost basis)
- **SOPR** — spent-output profit ratio (>1 = spending in profit)
- **NUPL** — net unrealized profit/loss (aggregate holder P&L)
- **realized_price** — aggregate on-chain cost basis (USD/BTC)
- **RHODL** — realized HODL ratio (cycle top/bottom oscillator)
- **exchange_balance** — BTC held on exchanges (liquidity/distribution proxy)

The Glassnode Light API caps history and daily calls, so we throttle harder and
respect `since` for incremental pulls. **Without a key**, plausible synthetic
paths anchored to the BTC cycle are generated so the full pipeline runs — set
`GLASSNODE_API_KEY` for the real thing.


In [4]:
# Glassnode endpoint path -> (our metric name, synthetic base / character)
GLASSNODE_METRICS: dict[str, dict[str, Any]] = {
    "market/mvrv":                        {"metric": "mvrv",             "base": 1.6,  "scale": 1.1},
    "indicators/sopr":                    {"metric": "sopr",             "base": 1.0,  "scale": 0.12},
    "indicators/nupl":                    {"metric": "nupl",             "base": 0.0,  "scale": 0.45},
    "market/price-realized-usd":          {"metric": "realized_price",   "base": 0.0,  "scale": 1.0},
    "indicators/rhodl-ratio":             {"metric": "rhodl",            "base": 1500, "scale": 4000},
    "transactions/balance-exchanges":     {"metric": "exchange_balance", "base": 2.5e6,"scale": 0.4e6},
}
GLASSNODE_API = "https://api.glassnode.com"


def fetch_glassnode(path: str, asset: str = "BTC", since: int | None = None) -> list[tuple[date, float]]:
    """Fetch a Glassnode metric series. Requires GLASSNODE_API_KEY."""
    params: dict[str, str | int] = {"a": asset, "api_key": GLASSNODE_API_KEY}
    if since is not None:
        params["s"] = since
    with httpx.Client(timeout=30.0) as client:
        resp = client.get(f"{GLASSNODE_API}/v1/metrics/{path}", params=params)
    if resp.status_code == 429:
        time.sleep(60)
        resp.raise_for_status()
    resp.raise_for_status()
    data = resp.json()
    return [(datetime.fromtimestamp(int(pt["t"]), tz=UTC).date(), float(pt["v"]))
            for pt in data if pt.get("v") is not None]


def sync_glassnode(force: bool = False) -> None:
    """Keyed, throttled sync of Glassnode metrics. No-op (synthetic later) without a key."""
    if not GLASSNODE_API_KEY:
        print("GLASSNODE_API_KEY not set — skipping live fetch (synthetic fallback used downstream).")
        return
    now = datetime.now(UTC)
    print(f"Syncing Glassnode ({len(GLASSNODE_METRICS)} metrics) "
          f"{'[FORCE]' if force else '[stale-only]'} ...")
    for path, meta in GLASSNODE_METRICS.items():
        metric = meta["metric"]
        latest_at, last_refresh = store.get_state(metric, "glassnode")
        if not force and last_refresh is not None:
            age_h = (now - last_refresh).total_seconds() / 3600
            if age_h < REFRESH_STALE_HOURS:
                print(f"  {metric:>18}: fresh ({age_h:.1f}h ago) — skip")
                continue
        since = int(datetime.fromisoformat(str(latest_at)).timestamp()) + 86400 if latest_at else None
        try:
            rows = fetch_glassnode(path, since=since)
        except Exception as exc:
            print(f"  {metric:>18}: FAILED ({exc})")
            continue
        n = store.upsert_series(metric, rows, "glassnode")
        latest = max(d for d, _ in rows) if rows else None
        store.upsert_state(metric, "glassnode", latest)
        print(f"  {metric:>18}: {n:>5} rows  up to {latest}")
        time.sleep(GLASSNODE_REQUEST_DELAY)
    print("Glassnode sync done.")


sync_glassnode()


GLASSNODE_API_KEY not set — skipping live fetch (synthetic fallback used downstream).


## 2. Load crypto panel → weekly BTC frame

On-chain daily series are collapsed to weekly to match the lowest common
frequency (Glassnode daily) and to align with the `Macro.ipynb` cadence — enough
resolution for lead/lag detection without daily noise.


In [5]:
panel = load_daily_panel(DB_PATH)
print(f"Full panel: {panel.shape}  symbols: {panel['symbol'].n_unique()}")

btc = panel.filter(pl.col("symbol") == "BTC").sort("date").unique(subset=["date"])
print(f"BTC daily: {btc.height} rows  {btc['date'].min()} -> {btc['date'].max()}")

btc_weekly = (
    btc.with_columns(pl.col("date").dt.truncate(WEEKLY_FREQ).alias("week"))
    .group_by("week")
    .agg(
        pl.col("close").last().alias("close"),
        pl.col("volume").sum().alias("volume"),
    )
    .sort("week")
    .with_columns(
        np.log(pl.col("close")).alias("log_close"),
        (pl.col("close") / pl.col("close").shift(1) - 1.0).alias("wk_return"),
    )
)
btc_weekly = btc_weekly.with_columns(
    (pl.col("log_close") - pl.col("log_close").cum_max()).alias("log_drawdown")
)
print(f"BTC weekly: {btc_weekly.height} rows  {btc_weekly['week'].min()} -> {btc_weekly['week'].max()}")
btc_weekly.tail(3)


Full panel: (70344, 8)  symbols: 47
BTC daily: 4001 rows  2015-07-20 -> 2026-07-02
BTC weekly: 572 rows  2015-07-20 -> 2026-06-29


week,close,volume,log_close,wk_return,log_drawdown
date,f64,f64,f64,f64,f64
2026-06-15,63235.63,40738.880507,11.054623,-0.037606,-0.669542
2026-06-22,59474.01,68392.996427,10.993295,-0.059486,-0.73087
2026-06-29,61646.26,38666.497833,11.029168,0.036524,-0.694997


## 3. blockchain.info series → weekly on-chain frame

Load each keyless series from the local store and asof-join onto the BTC weekly
spine (backward fill for sub-weekly refresh). Forward-fill the sparse monthly
series (supply is monthly).


In [6]:
week_spine = btc_weekly.select(pl.col("week").alias("date")).sort("date")

bc_weekly = week_spine
for _chart, meta in BLOCKCHAIN_METRICS.items():
    metric = meta["metric"]
    df = store.load_series(metric).sort("date")
    if df.is_empty():
        print(f"  WARN: no stored data for {metric}")
        df = pl.DataFrame({"date": week_spine["date"].to_list(), "value": [np.nan] * week_spine.height})
    df = df.rename({"value": metric})
    bc_weekly = bc_weekly.join_asof(df, on="date", strategy="backward")

bc_weekly = bc_weekly.with_columns(*[
    pl.col(m).forward_fill() for m in [m["metric"] for m in BLOCKCHAIN_METRICS.values()]
])
print(f"blockchain.info weekly frame: {bc_weekly.shape}")
bc_weekly.tail(3)


blockchain.info weekly frame: (572, 11)


date,hashrate,difficulty,miner_revenue_usd,fees_usd,active_addresses,tx_count,transfer_volume_usd,market_cap,supply,cost_per_tx_pct
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-06-15,9.0812e8,1.2512e14,2.5758e7,222065.651465,515594.0,647883.0,6.1233e9,1.3342e12,2.0043e7,0.524016
2026-06-22,9.6883e8,1.2493e14,2.8231e7,162753.338842,409924.0,779461.0,4.4456e9,1.2827e12,2.0046e7,0.752511
2026-06-29,9.6262e8,1.2493e14,2.5523e7,214552.53018,416739.0,616427.0,2.3571e9,1.2006e12,2.0049e7,1.432092


## 4. Glassnode valuation indicators → weekly

Load stored Glassnode series when present. For any metric not in the store
(no key), generate a **plausible synthetic path** anchored to the BTC weekly
log-close cycle so the full pipeline runs end-to-end. The synthetics are
deliberately cycle-correlated so downstream lead/lag and regime logic produce
sensible, non-degenerate results — but **they are not real signals**; set
`GLASSNODE_API_KEY` for research.


In [7]:
def synthetic_glassnode(dates: pl.Series, metric: str, btc_log_close: pl.Series) -> pl.DataFrame:
    """Plausible cycle-anchored synthetic for a missing Glassnode metric."""
    meta = next(m for k, m in GLASSNODE_METRICS.items() if m["metric"] == metric)
    base, scale = float(meta["base"]), float(meta["scale"])
    rng = np.random.default_rng(abs(hash(metric)) % (2**32))
    n = dates.len()
    t = np.arange(n)
    lc = btc_log_close.to_numpy()
    lc_norm = (lc - np.nanmean(lc)) / (np.nanstd(lc) + 1e-12)
    cyc = 0.85 * np.sin(2 * np.pi * t / 208.0)   # ~4y halving cycle
    noise = rng.normal(0, 0.04, n)
    if metric == "mvrv":
        val = base + scale * (0.6 * lc_norm + 0.4 * cyc) + noise * scale * 0.15
        val = np.clip(val, 0.6, 7.0)
    elif metric == "sopr":
        val = base + scale * (0.7 * lc_norm + 0.3 * cyc) + noise * scale * 0.3
        val = np.clip(val, 0.7, 1.25)
    elif metric == "nupl":
        val = base + scale * (0.65 * lc_norm + 0.35 * cyc) + noise * scale * 0.2
        val = np.clip(val, -0.2, 0.8)
    elif metric == "realized_price":
        # smooth lagged tracker of price (USD), exp-scaled like real BTC
        val = np.exp(np.nanmean(lc) - 1.5 + 0.7 * lc_norm + 0.3 * cyc + np.cumsum(noise) * 0.02)
        val = np.clip(val, 100.0, None)
    elif metric == "rhodl":
        val = base + scale * (0.5 * lc_norm + 0.5 * cyc) + noise * scale * 0.1
        val = np.clip(val, 50.0, None)
    elif metric == "exchange_balance":
        val = base + scale * (0.3 - 0.5 * lc_norm + 0.2 * cyc) + np.cumsum(noise) * scale * 0.05
        val = np.clip(val, 1.0e6, None)
    else:
        val = base + scale * lc_norm
    return pl.DataFrame({"date": dates.to_list(), metric: val}).sort("date")


gn_weekly = week_spine
present = set(store.metrics_present())
for _path, meta in GLASSNODE_METRICS.items():
    metric = meta["metric"]
    if metric in present:
        df = store.load_series(metric).sort("date").rename({"value": metric})
    else:
        df = synthetic_glassnode(week_spine["date"], metric, btc_weekly["log_close"])
    gn_weekly = gn_weekly.join_asof(df, on="date", strategy="backward")

gn_weekly = gn_weekly.with_columns(*[
    pl.col(m["metric"]).forward_fill() for m in GLASSNODE_METRICS.values()
])
gn_live = {m["metric"] for m in GLASSNODE_METRICS.values() if m["metric"] in present}
print(f"Glassnode weekly frame: {gn_weekly.shape}")
print(f"Live Glassnode metrics: {sorted(gn_live) or 'none (all synthetic)'}")
gn_weekly.tail(3)


Glassnode weekly frame: (572, 7)
Live Glassnode metrics: none (all synthetic)


date,mvrv,sopr,nupl,realized_price,rhodl,exchange_balance
date,f64,f64,f64,f64,f64,f64
2026-06-15,1.863954,1.051119,0.1429,4147.5854,1728.392834,2.3789e6
2026-06-22,1.83398,1.045202,0.136821,4043.75032,1652.112498,2.3868e6
2026-06-29,1.838973,1.049349,0.143898,4103.92674,1714.390669,2.3831e6


## 5. On-chain feature engineering

Combine the keyless fundamentals with the Glassnode valuation ratios and derive
the interpretable, popular on-chain indicators:

| Feature | Construction | Signal |
|---|---|---|
| **hashprice** | `miner_revenue_usd / hashrate` | miner revenue per TH/day → profitability |
| **puell_multiple** | `miner_revenue_usd / 365d-avg(miner_revenue_usd)` | miner stress cycle (low = capitulation) |
| **fee_rev_share** | `fees_usd / miner_revenue_usd` | fee dependence / tx demand |
| **active_addr_yoy** | 52w log growth of active addresses | adoption momentum |
| **tx_count_yoy** | 52w log growth of tx count | activity momentum |
| **transfer_velocity** | `transfer_volume_usd / market_cap` | coin velocity / whale-flow proxy |
| **hashrate_yoy** | 52w log growth of hashrate | security / miner commitment |
| **mvrv**, **sopr**, **nupl**, **rhodl** | (Glassnode) | cycle valuation |
| **exchange_balance_yoy** | 52w log change | supply leaving exchanges = accumulation |

Then merge BTC weekly onto the on-chain frame.


In [8]:
def z_expr(col: str) -> pl.Expr:
    return (pl.col(col) - pl.col(col).mean()) / pl.col(col).std()

# Merge keyless + Glassnode frames onto the weekly spine
oc = week_spine
for frame in (bc_weekly, gn_weekly):
    oc = oc.join(frame, on="date", how="left", suffix="_dup")
oc = oc.drop([c for c in oc.columns if c.endswith("_dup")])

oc = oc.with_columns(
    (pl.col("miner_revenue_usd") / (pl.col("hashrate") + 1e-9)).alias("hashprice"),
    (pl.col("fees_usd") / (pl.col("miner_revenue_usd") + 1e-9)).alias("fee_rev_share"),
    (pl.col("transfer_volume_usd") / (pl.col("market_cap") + 1e-9)).alias("transfer_velocity"),
).with_columns(
    (pl.col("miner_revenue_usd") / pl.col("miner_revenue_usd").rolling_mean(365 // 7)).alias("puell_multiple"),
    (np.log(pl.col("active_addresses")) - np.log(pl.col("active_addresses")).shift(LOOKBACK_YOY)).alias("active_addr_yoy"),
    (np.log(pl.col("tx_count") + 1.0) - np.log(pl.col("tx_count") + 1.0).shift(LOOKBACK_YOY)).alias("tx_count_yoy"),
    (np.log(pl.col("hashrate")) - np.log(pl.col("hashrate")).shift(LOOKBACK_YOY)).alias("hashrate_yoy"),
    (np.log(pl.col("exchange_balance")) - np.log(pl.col("exchange_balance")).shift(LOOKBACK_YOY)).alias("exchange_balance_yoy"),
).with_columns(
    (pl.col("puell_multiple") - pl.col("puell_multiple").shift(MOM_LOOKBACK)).alias("puell_mom"),
    (pl.col("mvrv") - pl.col("mvrv").shift(MOM_LOOKBACK)).alias("mvrv_mom"),
    (pl.col("sopr") - pl.col("sopr").shift(MOM_LOOKBACK)).alias("sopr_mom"),
    (pl.col("nupl") - pl.col("nupl").shift(MOM_LOOKBACK)).alias("nupl_mom"),
)

oc = oc.join(
    btc_weekly.select("week", "close", "log_close", "wk_return", "log_drawdown"),
    left_on="date", right_on="week", how="left",
)
oc = oc.drop_nulls(subset=["close", "hashprice", "puell_multiple", "mvrv"])
print(f"On-chain + BTC frame: {oc.shape}  {oc['date'].min()} -> {oc['date'].max()}")
oc.select("date", "hashprice", "puell_multiple", "active_addr_yoy",
          "transfer_velocity", "mvrv", "sopr", "nupl", "close").tail(5)


On-chain + BTC frame: (521, 33)  2016-07-11 -> 2026-06-29


date,hashprice,puell_multiple,active_addr_yoy,transfer_velocity,mvrv,sopr,nupl,close
date,f64,f64,f64,f64,f64,f64,f64,f64
2026-06-01,0.033511,0.744824,-0.115782,0.00247,1.861241,1.049791,0.142722,63302.73
2026-06-08,0.031265,0.615091,-0.103343,0.001757,1.875619,1.051258,0.155596,65706.62
2026-06-15,0.028364,0.608109,-0.096515,0.004589,1.863954,1.051119,0.1429,63235.63
2026-06-22,0.02914,0.66831,-0.105703,0.003466,1.83398,1.045202,0.136821,59474.01
2026-06-29,0.026514,0.610813,-0.059322,0.001963,1.838973,1.049349,0.143898,61646.26


## 6. On-chain cycle-valuation composite

A z-scored composite of the joint cycle-valuation signals — the on-chain
analogue of the Macro liquidity index. High = expensive/toppy; low = cheap/
capitulation:

**C = z(MVRV) + z(SOPR) + z(NUPL) + z(RHODL) − z(Puell)**

MVRV/SOPR/NUPL/RHODL rise into euphoria; a high Puell (miner revenue rich vs.
average) is a distribution risk. The composite level marks cycle positioning;
its **12-week momentum** sign defines the bullish/bearish regime and its
sign-changes are the trend turns we try to predict.


In [9]:
oc = oc.with_columns(
    (z_expr("mvrv") + z_expr("sopr") + z_expr("nupl") + z_expr("rhodl") - z_expr("puell_multiple")).alias("cycle_raw")
)
mu, sd = float(oc["cycle_raw"].mean()), float(oc["cycle_raw"].std())
oc = oc.with_columns(
    ((pl.col("cycle_raw") - mu) / (sd if sd > 1e-12 else 1.0)).alias("cycle_index"),
).with_columns(
    (pl.col("cycle_index") - pl.col("cycle_index").shift(MOM_LOOKBACK)).alias("cycle_mom"),
).with_columns(
    ((pl.col("cycle_mom") > 0).cast(pl.Int8)).alias("regime"),
)
print("On-chain cycle-valuation composite summary:")
print(oc.select("cycle_index", "cycle_mom", "regime").describe())


On-chain cycle-valuation composite summary:
shape: (9, 4)
┌────────────┬─────────────┬───────────┬──────────┐
│ statistic  ┆ cycle_index ┆ cycle_mom ┆ regime   │
│ ---        ┆ ---         ┆ ---       ┆ ---      │
│ str        ┆ f64         ┆ f64       ┆ f64      │
╞════════════╪═════════════╪═══════════╪══════════╡
│ count      ┆ 521.0       ┆ 509.0     ┆ 509.0    │
│ null_count ┆ 0.0         ┆ 12.0      ┆ 12.0     │
│ mean       ┆ -1.9775e-16 ┆ 0.042086  ┆ 0.530452 │
│ std        ┆ 1.0         ┆ 0.242846  ┆ 0.499563 │
│ min        ┆ -2.421483   ┆ -1.060682 ┆ 0.0      │
│ 25%        ┆ -1.014125   ┆ -0.136639 ┆ 0.0      │
│ 50%        ┆ 0.107491    ┆ 0.014379  ┆ 1.0      │
│ 75%        ┆ 0.635849    ┆ 0.207177  ┆ 1.0      │
│ max        ┆ 1.785298    ┆ 1.293747  ┆ 1.0      │
└────────────┴─────────────┴───────────┴──────────┘


In [10]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=oc["date"], y=oc["cycle_index"], name="cycle composite",
                         line=dict(color="#39ff14", width=1.8), yaxis="y"))
fig.add_trace(go.Scatter(x=oc["date"], y=np.exp(oc["log_close"]), name="BTC",
                         line=dict(color="#f7931a", width=1.5), yaxis="y2"))
fig.update_layout(
    title="On-chain cycle-valuation composite vs BTC (weekly)",
    template="plotly_dark", height=450,
    yaxis=dict(title="cycle z"),
    yaxis2=dict(title="BTC $", overlaying="y", side="right", type="log"),
    legend=dict(orientation="h", y=1.08),
)
fig.show()
corr = float(oc.select(pl.corr("cycle_index", "log_close")).item())
print(f"corr(cycle_index, log BTC) = {corr:.3f}")


corr(cycle_index, log BTC) = 0.858


## 7. Lead/lag analysis — which on-chain signals lead BTC?

Cross-correlation of each signal vs. **forward 13-week** BTC log return (lag>0 =
signal leads price) plus Granger causality on weekly returns. The signals that
historically turn *first* feed the early-warning composite in §8.


In [11]:
oc = oc.with_columns(
    (pl.col("log_close").shift(-13) - pl.col("log_close")).alias("fwd_13w")
)

lead_df = oc.select(
    "date", "cycle_index", "hashprice", "puell_multiple", "fee_rev_share",
    "active_addr_yoy", "tx_count_yoy", "hashrate_yoy", "transfer_velocity",
    "mvrv", "sopr", "nupl", "rhodl", "exchange_balance_yoy", "fwd_13w", "wk_return",
).drop_nulls()


def cross_corr(x: np.ndarray, y: np.ndarray, lags: range) -> list[tuple[int, float]]:
    """corr(x[t-lag], y[t]); lag>0 => x leads y."""
    out = []
    n = len(y)
    for lag in lags:
        if lag < 0:
            a, b = x[-lag:], y[: n + lag]
        else:
            a, b = x[: n - lag], y[lag:]
        if len(a) > 20:
            out.append((lag, float(np.corrcoef(a, b)[0, 1])))
    return out


fwd13 = lead_df["fwd_13w"].to_numpy()
btc_ret = lead_df["wk_return"].to_numpy()
signals = {
    "cycle composite":      lead_df["cycle_index"].to_numpy(),
    "hashprice":            lead_df["hashprice"].to_numpy(),
    "puell multiple":       lead_df["puell_multiple"].to_numpy(),
    "fee rev share":        lead_df["fee_rev_share"].to_numpy(),
    "active addr yoy":      lead_df["active_addr_yoy"].to_numpy(),
    "tx count yoy":         lead_df["tx_count_yoy"].to_numpy(),
    "hashrate yoy":         lead_df["hashrate_yoy"].to_numpy(),
    "transfer velocity":    lead_df["transfer_velocity"].to_numpy(),
    "mvrv":                 lead_df["mvrv"].to_numpy(),
    "sopr":                 lead_df["sopr"].to_numpy(),
    "nupl":                 lead_df["nupl"].to_numpy(),
    "rhodl":                lead_df["rhodl"].to_numpy(),
    "exchange bal yoy":     lead_df["exchange_balance_yoy"].to_numpy(),
}

lags = range(LEAD_MIN, LEAD_MAX + 1)
lead_results: dict[str, dict[str, Any]] = {}
print("Cross-correlation with FORWARD 13-week BTC log return (lag>0 = signal leads):\n")
print(f"  {'signal':>20} {'lead lag':>8} {'lead corr':>10} {'@lag0':>8}")
for name, sig in signals.items():
    cc = cross_corr(sig, fwd13, lags)
    lead_only = [(lg, c) for lg, c in cc if lg > 0]
    best_lead = max(lead_only, key=lambda t: abs(t[1])) if lead_only else (0, 0.0)
    lag0 = next((c for lg, c in cc if lg == 0), 0.0)
    lead_results[name] = {"lead_lag": best_lead[0], "lead_corr": best_lead[1], "cc": cc}
    print(f"  {name:>20} {best_lead[0]:>7}w {best_lead[1]:>+9.2f} {lag0:>+8.2f}")


Cross-correlation with FORWARD 13-week BTC log return (lag>0 = signal leads):

                signal lead lag  lead corr    @lag0
       cycle composite       7w     -0.16    -0.15
             hashprice      26w     -0.09    +0.05
        puell multiple      26w     -0.20    -0.03
         fee rev share       1w     +0.19    +0.19
       active addr yoy       1w     +0.22    +0.24
          tx count yoy       1w     +0.27    +0.26
          hashrate yoy      26w     +0.20    -0.03
     transfer velocity       4w     +0.30    +0.28
                  mvrv      26w     -0.22    -0.17
                  sopr      26w     -0.28    -0.24
                  nupl      26w     -0.20    -0.17
                 rhodl      26w     -0.13    -0.07
      exchange bal yoy      17w     +0.35    +0.17


In [12]:
lag_axis = list(range(LEAD_MIN, LEAD_MAX + 1))
mat, names = [], []
for name, r in lead_results.items():
    cc_map = dict(r["cc"])
    mat.append([cc_map.get(lg, np.nan) for lg in lag_axis])
    names.append(name)

fig = go.Figure(data=go.Heatmap(
    z=mat, x=lag_axis, y=names, colorscale="RdBu", zmid=0,
    colorbar=dict(title="corr"),
))
fig.update_layout(
    title="Cross-correlation: on-chain signal[t-lag] vs FORWARD 13-week BTC return (lag>0 = leads)",
    xaxis_title="lag (weeks)", template="plotly_dark", height=500,
)
fig.show()


In [13]:
print("Granger causality (on-chain signal -> BTC weekly return), maxlag=12:\n")
for name in ["cycle composite", "puell multiple", "hashprice", "mvrv", "nupl", "active addr yoy"]:
    sig = signals[name]
    sig_diff = np.diff(sig, prepend=sig[0])
    try:
        res = grangercausalitytests(
            np.column_stack([btc_ret, sig_diff]), maxlag=12, verbose=False
        )
        pvals = [res[lag][0]["ssr_ftest"][1] for lag in range(1, 13)]
        best_lag = int(np.argmin(pvals) + 1)
        print(f"  {name:>20}: min p = {min(pvals):.3f} at lag {best_lag}w")
    except Exception as exc:
        print(f"  {name:>20}: failed ({exc})")


Granger causality (on-chain signal -> BTC weekly return), maxlag=12:

       cycle composite: min p = 0.224 at lag 5w
        puell multiple: min p = 0.241 at lag 5w
             hashprice: min p = 0.001 at lag 3w
                  mvrv: min p = 0.194 at lag 5w
                  nupl: min p = 0.141 at lag 9w
       active addr yoy: min p = 0.124 at lag 6w


## 8. Early-warning composite + regime backtest

An **early-warning** momentum composite from the signals that historically turn
*first* (negative-Puell momentum = miner relief, MVRV momentum = re-rating,
active-address momentum = adoption resumption). When it crosses up from below
zero we flag a **bullish trigger**.

Then backtest **forward BTC returns** (1m / 3m / 6m / 12m) conditional on
(i) the cycle regime and (ii) early-warn bullish triggers.


In [14]:
oc = oc.with_columns(
    (-z_expr("puell_mom")).alias("_c1"),
    z_expr("mvrv_mom").alias("_c2"),
    z_expr("active_addr_yoy").alias("_c3"),
    z_expr("sopr_mom").alias("_c4"),
).with_columns(
    (pl.col("_c1") + pl.col("_c2") + pl.col("_c3") + pl.col("_c4")).alias("early_warn_raw")
).drop(["_c1", "_c2", "_c3", "_c4"])

mu, sd = float(oc["early_warn_raw"].mean()), float(oc["early_warn_raw"].std())
oc = oc.with_columns(
    ((pl.col("early_warn_raw") - mu) / (sd if sd > 1e-12 else 1.0)).alias("early_warn")
)

ew = oc["early_warn"].to_numpy()
ci = oc["cycle_index"].to_numpy()
mask = ~(np.isnan(ew) | np.isnan(ci))
cc_ew = cross_corr(ew[mask], ci[mask], range(LEAD_MIN, LEAD_MAX + 1))
best = max(cc_ew, key=lambda t: abs(t[1]))
print(f"Early-warn leads cycle level by ~{best[0]}w  (corr={best[1]:+.2f})")
print("(positive lag = early-warn turns before the cycle-valuation level)")


Early-warn leads cycle level by ~-26w  (corr=-0.38)
(positive lag = early-warn turns before the cycle-valuation level)


In [15]:
oc = oc.with_columns(
    (pl.col("regime") != pl.col("regime").shift(1)).alias("regime_flip")
)
flips = oc.filter(
    pl.col("regime_flip") & pl.col("regime").is_not_null()
    & pl.col("regime").shift(1).is_not_null()
).select("date", "regime")
print(f"Detected {flips.height} on-chain regime flips")
print("Most recent flips to bullish (regime=1):")
print(flips.filter(pl.col("regime") == 1).tail(8))


Detected 82 on-chain regime flips
Most recent flips to bullish (regime=1):
shape: (8, 2)
┌────────────┬────────┐
│ date       ┆ regime │
│ ---        ┆ ---    │
│ date       ┆ i8     │
╞════════════╪════════╡
│ 2024-09-02 ┆ 1      │
│ 2024-09-23 ┆ 1      │
│ 2024-10-14 ┆ 1      │
│ 2024-11-04 ┆ 1      │
│ 2024-12-30 ┆ 1      │
│ 2025-01-13 ┆ 1      │
│ 2025-06-23 ┆ 1      │
│ 2026-05-04 ┆ 1      │
└────────────┴────────┘


In [16]:
first_reg = oc.filter(pl.col("regime").is_not_null())["regime"][0]
spans: list[tuple[Any, Any, int | None]] = []
start = oc["date"].min()
prev_reg: int | None = int(first_reg)
for d, r in zip(flips["date"].to_list(), flips["regime"].to_list(), strict=True):
    spans.append((start, d, prev_reg))
    start = d
    prev_reg = int(r)
spans.append((start, oc["date"].max(), prev_reg))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=oc["date"], y=np.exp(oc["log_close"]), name="BTC",
    line=dict(color="#f7931a", width=1.5),
))
for s, e, r in spans:
    if r == 1:
        fig.add_vrect(x0=s, x1=e, fillcolor="rgba(57,255,20,0.10)", line_width=0)
fig.update_layout(
    title="BTC with on-chain bullish regimes shaded (green)",
    yaxis_type="log", template="plotly_dark", height=450,
)
fig.show()


In [17]:
for h in FWD_HORIZONS_WK:
    oc = oc.with_columns(
        (pl.col("log_close").shift(-h) - pl.col("log_close")).alias(f"fwd_{h}w")
    )

bt_rows: list[dict[str, Any]] = []
print("Forward BTC log returns by on-chain regime:\n")
print(f"  {'horizon':>8} {'regime':>9} {'n':>5} {'mean':>8} {'median':>8} {'p25':>8} {'p75':>8} {'pos%':>6}")
for h in FWD_HORIZONS_WK:
    col = f"fwd_{h}w"
    for r, label in [(1, "bull"), (0, "bear")]:
        s = oc.filter(pl.col("regime") == r).select(col).drop_nulls().to_series().to_numpy()
        if len(s) == 0:
            continue
        row = {
            "horizon": h, "regime": label, "n": len(s),
            "mean": float(np.mean(s)), "median": float(np.median(s)),
            "p25": float(np.percentile(s, 25)), "p75": float(np.percentile(s, 75)),
            "pos_pct": float((s > 0).mean()),
        }
        bt_rows.append(row)
        print(f"  {h:>7}w {label:>9} {len(s):>5} {row['mean']:+.3f} {row['median']:+.3f} "
              f"{row['p25']:+.3f} {row['p75']:+.3f} {row['pos_pct']:>5.0%}")


Forward BTC log returns by on-chain regime:

   horizon    regime     n     mean   median      p25      p75   pos%
        4w      bull   270 +0.057 +0.039 -0.049 +0.192   59%
        4w      bear   235 +0.013 +0.012 -0.111 +0.115   52%
       13w      bull   269 +0.155 +0.118 -0.105 +0.373   63%
       13w      bear   227 +0.078 +0.015 -0.238 +0.374   52%
       26w      bull   269 +0.281 +0.260 -0.055 +0.526   70%
       26w      bear   214 +0.183 +0.068 -0.468 +0.670   53%
       52w      bull   269 +0.544 +0.560 +0.087 +0.928   79%
       52w      bear   188 +0.347 +0.255 -0.361 +0.836   56%


In [18]:
oc = oc.with_columns(
    ((pl.col("early_warn") > 0) & (pl.col("early_warn").shift(1) <= 0)).alias("ew_bull_trigger")
)
trig = oc.filter(pl.col("ew_bull_trigger"))
print(f"Early-warn bullish triggers: {trig.height}\n")
print(f"  {'horizon':>8} {'n':>5} {'mean':>8} {'median':>8} {'pos%':>6}")
for h in FWD_HORIZONS_WK:
    s = trig.select(f"fwd_{h}w").drop_nulls().to_series().to_numpy()
    if len(s):
        print(f"  {h:>7}w {len(s):>5} {np.mean(s):+.3f} {np.median(s):+.3f} {(s > 0).mean():>5.0%}")


Early-warn bullish triggers: 26

   horizon     n     mean   median   pos%
        4w    26 +0.041 +0.022   54%
       13w    26 +0.202 +0.087   54%
       26w    26 +0.457 +0.372   81%
       52w    25 +0.633 +0.729   68%


## 9. Probit model — P(bullish regime in ~90 days)

Predict the on-chain regime ~13 weeks ahead from a compact set of standardized
on-chain features. Scored walk-forward with the Brier score (vs. the naive
base-rate Brier) so calibration is honest.


In [19]:
horizon = PROBIT_HORIZON_WK
oc = oc.with_columns(
    pl.col("regime").shift(-horizon).cast(pl.Float64).alias("regime_fwd")
)

feat_cols = ["puell_multiple", "hashprice", "active_addr_yoy", "tx_count_yoy",
             "transfer_velocity", "mvrv", "sopr", "nupl", "early_warn"]
fit_df = oc.select(["regime_fwd"] + feat_cols).drop_nulls()

y_p = fit_df["regime_fwd"].to_numpy()
X_p = fit_df.select(feat_cols).to_numpy().astype(float)
mu_x = X_p.mean(axis=0)
sd_x = X_p.std(axis=0)
sd_x[sd_x < 1e-12] = 1.0
Xp_c = np.column_stack([np.ones(len(X_p)), (X_p - mu_x) / sd_x])

probit = sm.Probit(y_p, Xp_c).fit(method="bfgs", maxiter=1000, disp=0)
feat_names = ["const"] + feat_cols
print("Probit coefficients (standardized features):")
print(f"  {'feature':>20} {'coef':>9}")
for nm, cf in zip(feat_names, probit.params, strict=True):
    print(f"  {nm:>20} {cf:+.3f}")
pos_rate = float(y_p.mean())
print(f"\nPseudo R2 = {probit.prsquared:.3f}  n = {int(probit.nobs)}  base rate = {pos_rate:.0%}")

oos_brier: list[float] = []
min_train = 80
for o in range(min_train, len(Xp_c) - horizon, 13):
    if len(np.unique(y_p[:o])) < 2:
        continue
    m = sm.Probit(y_p[:o], Xp_c[:o]).fit(method="bfgs", maxiter=500, disp=0)
    p = float(m.predict(Xp_c[o : o + 1])[0])
    oos_brier.append((p - y_p[o]) ** 2)
naive_brier = pos_rate * (1 - pos_rate)
print(f"Walk-forward Brier: {np.mean(oos_brier):.3f} over {len(oos_brier)} origins "
      f"(naive base-rate Brier = {naive_brier:.3f})")

p_bull = float(probit.predict(Xp_c[-1:])[0])
print(f"\nCurrent P(bullish on-chain regime in {horizon}w): {p_bull:.0%}")


Probit coefficients (standardized features):
               feature      coef
                 const +0.147
        puell_multiple +0.791
             hashprice -0.737
       active_addr_yoy -0.522
          tx_count_yoy +0.622
     transfer_velocity -0.073
                  mvrv +5.518
                  sopr -0.555
                  nupl -5.662
            early_warn +0.382

Pseudo R2 = 0.388  n = 496  base rate = 53%
Walk-forward Brier: 0.167 over 31 origins (naive base-rate Brier = 0.249)

Current P(bullish on-chain regime in 13w): 49%


## 10. Dashboard


In [20]:
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    row_heights=[0.45, 0.35, 0.20],
    subplot_titles=("BTC price (log) with on-chain regimes",
                    "Cycle-valuation level vs early-warning momentum",
                    "12w cycle momentum (regime signal)"),
)
fig.add_trace(go.Scatter(x=oc["date"], y=np.exp(oc["log_close"]), name="BTC",
                         line=dict(color="#f7931a", width=1.6)), row=1, col=1)
for s, e, r in spans:
    if r == 1:
        fig.add_vrect(x0=s, x1=e, fillcolor="rgba(57,255,20,0.10)",
                      line_width=0, row=1, col=1)
fig.add_trace(go.Scatter(x=oc["date"], y=oc["cycle_index"], name="cycle composite",
                         line=dict(color="#39ff14", width=1.6)), row=2, col=1)
fig.add_trace(go.Scatter(x=oc["date"], y=oc["early_warn"], name="early-warn (leading)",
                         line=dict(color="#6ea8ff", width=1.2, dash="dot")), row=2, col=1)
fig.add_trace(go.Bar(x=oc["date"], y=oc["cycle_mom"], name="cycle momentum (12w)",
                     marker_color="#888", showlegend=False), row=3, col=1)
fig.add_hline(y=0, line_color="gray", line_width=0.5, row=3, col=1)
fig.update_layout(
    template="plotly_dark", height=720,
    legend=dict(orientation="h", y=1.04),
    yaxis_type="log", yaxis2=dict(title="z"), yaxis3=dict(title="z"),
)
fig.show()


In [21]:
bt = pl.DataFrame(bt_rows)
fig2 = go.Figure()
for label, color in [("bull", "#39ff14"), ("bear", "#ff5555")]:
    sub = bt.filter(pl.col("regime") == label).sort("horizon")
    rows = sub.to_dicts()
    fig2.add_trace(go.Bar(
        x=[f"{r['horizon']}w" for r in rows], y=[r["mean"] for r in rows],
        name=label, marker_color=color,
        error_y=dict(type="data",
                     array=[r["p75"] - r["mean"] for r in rows],
                     arrayminus=[r["mean"] - r["p25"] for r in rows]),
    ))
fig2.add_hline(y=0, line_color="gray")
fig2.update_layout(
    title="Forward BTC log return by on-chain regime (mean +/- IQR)",
    template="plotly_dark", barmode="group", height=400,
    yaxis_title="forward log return",
)
fig2.show()


## 11. Snapshot & caveats


In [22]:
latest = oc.tail(1)
print("=" * 72)
print("ON-CHAIN BTC SIGNAL SNAPSHOT")
print("=" * 72)
print(f"Latest date:              {latest['date'][0]}")
print(f"Cycle-valuation index:    {latest['cycle_index'][0]:+.2f}")
print(f"Early-warn (leading):     {latest['early_warn'][0]:+.2f}")
print(f"Current regime:           {'BULLISH' if latest['regime'][0] else 'BEARISH'}")
print(f"P(bullish in 13w):        {p_bull:.0%}")
print(f"Hashprice (USD/TH/day):   {latest['hashprice'][0]:.4f}")
print(f"Puell multiple:           {latest['puell_multiple'][0]:.2f}")
print(f"Active addr YoY:          {latest['active_addr_yoy'][0] * 100:+.1f}%")
print(f"Transfer velocity:        {latest['transfer_velocity'][0]:.3f}")
print(f"MVRV:                     {latest['mvrv'][0]:.2f}")
print(f"SOPR:                     {latest['sopr'][0]:.3f}")
print(f"NUPL:                     {latest['nupl'][0]:.3f}")
print()
print("Leading signals (corr with FORWARD 13-week BTC return, lag>0 = leads):")
for name, r in sorted(lead_results.items(), key=lambda kv: -abs(kv[1]["lead_corr"])):
    print(f"  {name:>20}: lead {r['lead_lag']:>3}w  corr {r['lead_corr']:+.2f}")
print()
print(f"Probit pseudo-R2: {probit.prsquared:.3f}  OOS Brier: {np.mean(oos_brier):.3f}")
print()
print(f"Glassnode metrics live: {sorted(gn_live) or 'none (synthetic fallback)'}")
print("Store:", ONCHAIN_DB)
print(f"  metrics: {store.metrics_present()}")
store.close()


ON-CHAIN BTC SIGNAL SNAPSHOT
Latest date:              2026-06-29
Cycle-valuation index:    +0.39
Early-warn (leading):     -0.87
Current regime:           BEARISH
P(bullish in 13w):        49%
Hashprice (USD/TH/day):   0.0265
Puell multiple:           0.61
Active addr YoY:          -5.9%
Transfer velocity:        0.002
MVRV:                     1.84
SOPR:                     1.049
NUPL:                     0.144

Leading signals (corr with FORWARD 13-week BTC return, lag>0 = leads):
      exchange bal yoy: lead  17w  corr +0.35
     transfer velocity: lead   4w  corr +0.30
                  sopr: lead  26w  corr -0.28
          tx count yoy: lead   1w  corr +0.27
       active addr yoy: lead   1w  corr +0.22
                  mvrv: lead  26w  corr -0.22
          hashrate yoy: lead  26w  corr +0.20
                  nupl: lead  26w  corr -0.20
        puell multiple: lead  26w  corr -0.20
         fee rev share: lead   1w  corr +0.19
       cycle composite: lead   7w  corr -0.16
     

### Caveats

- **On-chain ≠ causal.** High R² / strong lead-lag reflect common cycle drivers
  (BTC price and on-chain activity co-move); coefficients are not causal effects.
  Interpret regimes as *structural state*, not trade triggers.
- **Glassnode fallback.** Without `GLASSNODE_API_KEY`, MVRV/SOPR/NUPL/RHODL/
  exchange_balance are *synthetic, cycle-anchored* paths — fine for pipeline
  development, **not research signals**. Set the key (Advanced/Light tier) for
  real valuation data.
- **blockchain.info is keyless & informal.** No published rate limit; the sync
  workflow is deliberately gentle (sequential, 1s spacing, 12h staleness gate,
  idempotent upserts). Don't raise `BC_INFO_REQUEST_DELAY` below ~0.5s.
- **Whale/flow proxies.** True whale cohort and exchange flow metrics are
  behind paid rails (Glassnode/CryptoQuant). `transfer_volume_usd` /
  `market_cap` (velocity) and `exchange_balance` (when keyed) are the closest
  free approximations here.
- **Puell / hashrate** are forward-filled weekly from daily series; miner
  revenue lags by ~1 day on the source.
- **Weekly cadence** smooths daily noise but reduces the number of independent
  walk-forward origins (BTC on-chain history ~10y); 12m-horizon coverage has
  few independent samples.
- **Point-in-time:** blockchain.info / Glassnode may revise recent points; for
  live trading use Glassnode's Point-in-Time endpoints.
- This is **research code**, not investment advice.
